In [10]:
# Önce eski cache'i sil
import shutil, os
cache_path = os.path.expanduser("~/.cache/huggingface/modules")
if os.path.exists(cache_path):
    shutil.rmtree(cache_path)
    print("Cache temizlendi")

# En güncel transformers'ı yükle
!pip install -q git+https://github.com/huggingface/transformers.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


# Fine-Tuning TinyLlama (Colab Ornek Defter)

Bu notebook su adimlari uctan uca gosterir:
1. Ornek veri yukleme
2. Model egitme (LoRA)
3. Cikti kaydetme

In [11]:
!nvidia-smi
!pip -q install -U transformers datasets accelerate peft trl bitsandbytes sentencepiece

Sat Mar 21 15:35:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             14W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1) Veri yukleme

Bu ornekte once kucuk bir ornek egitim verisi olusturup JSONL dosyasi olarak kaydediyoruz, sonra datasets ile yukluyoruz.

In [12]:
import json
import os
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

# Ornek egitim verisi
sample_rows = [
    {
        'instruction': 'Translate to English',
        'input': 'Merhaba dunya',
        'output': 'Hello world'
    },
    {
        'instruction': 'Summarize the text',
        'input': 'LLaMA is a family of language models by Meta.',
        'output': 'LLaMA is a Meta language model family.'
    },
    {
        'instruction': 'Answer the question',
        'input': 'What is 2 + 2?',
        'output': '4'
    }
]

os.makedirs('/content/data', exist_ok=True)
train_path = '/content/data/train.jsonl'

with open(train_path, 'w', encoding='utf-8') as f:
    for row in sample_rows:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

raw_dataset = load_dataset('json', data_files=train_path, split='train')
raw_dataset

CUDA available: True
GPU: Tesla T4


Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 3
})

## 2) Model hazirlama

MiniMax-M2.5 modelini 4-bit quantization ile yukleyip tokenizer ayarlarini yapiyoruz.
Not: Bu model Colab T4 gibi ortamlarda bellek sinirina takilabilir.

In [13]:
!pip install -q --upgrade transformers

In [14]:
!pip install -q git+https://github.com/huggingface/transformers.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [15]:
model_id = 'MiniMaxAI/MiniMax-M2.5'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True
)
model.config.use_cache = False

ImportError: cannot import name 'OutputRecorder' from 'transformers.utils.generic' (/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py)

In [ ]:
def format_example(x):
    return {
        'text': f"### Instruction:\n{x['instruction']}\n\n### Input:\n{x['input']}\n\n### Response:\n{x['output']}"
    }

dataset = raw_dataset.map(format_example)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM'
)

training_args = SFTConfig(
    output_dir='/content/tinyllama-lora-demo',
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy='no',
    max_seq_length=512
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    peft_config=peft_config,
    processing_class=tokenizer,
    dataset_text_field='text'
)

train_result = trainer.train()
train_result

In [ ]:
import zipfile

save_dir = '/content/output/tinyllama_lora_adapter'
os.makedirs(save_dir, exist_ok=True)

trainer.model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

metrics_path = '/content/output/train_metrics.json'
os.makedirs('/content/output', exist_ok=True)
with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump(train_result.metrics, f, ensure_ascii=False, indent=2)

zip_path = '/content/output/tinyllama_lora_adapter.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(save_dir):
        for file_name in files:
            full_path = os.path.join(root, file_name)
            arcname = os.path.relpath(full_path, save_dir)
            zf.write(full_path, arcname=arcname)

print('Kaydedilen klasor:', save_dir)
print('Metrik dosyasi:', metrics_path)
print('Zip dosyasi:', zip_path)

In [ ]:
prompt = '### Instruction:\nAnswer the question\n\n### Input:\nWhat is the capital of Turkiye?\n\n### Response:\n'
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

with torch.no_grad():
    output_ids = trainer.model.generate(**inputs, max_new_tokens=64, temperature=0.7)

print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

## 3) Model egitme

Veriyi instruction-input-response formatina cevirip LoRA ile kisa bir egitim yapiyoruz.

In [ ]:
def format_example(x):
    return {
        'text': f"### Instruction:\n{x['instruction']}\n\n### Input:\n{x['input']}\n\n### Response:\n{x['output']}"
    }

dataset = raw_dataset.map(format_example)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM'
)

training_args = SFTConfig(
    output_dir='/content/tinyllama-lora-demo',
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy='no',
    max_seq_length=512
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    peft_config=peft_config,
    processing_class=tokenizer,
    dataset_text_field='text'
)

train_result = trainer.train()
train_result

## 4) Cikti kaydetme

Egitim sonrasi LoRA adapter agirliklarini, tokenizer dosyalarini ve metrikleri dosyaya yaziyoruz.

In [ ]:
import zipfile

save_dir = '/content/output/tinyllama_lora_adapter'
os.makedirs(save_dir, exist_ok=True)

trainer.model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

metrics_path = '/content/output/train_metrics.json'
os.makedirs('/content/output', exist_ok=True)
with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump(train_result.metrics, f, ensure_ascii=False, indent=2)

zip_path = '/content/output/tinyllama_lora_adapter.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(save_dir):
        for file_name in files:
            full_path = os.path.join(root, file_name)
            arcname = os.path.relpath(full_path, save_dir)
            zf.write(full_path, arcname=arcname)

print('Kaydedilen klasor:', save_dir)
print('Metrik dosyasi:', metrics_path)
print('Zip dosyasi:', zip_path)

## 5) Hizli test

Egitimden sonra olusan modelle kisa bir uretim ornegi aliyoruz.

In [ ]:
prompt = '### Instruction:\nAnswer the question\n\n### Input:\nWhat is the capital of Turkiye?\n\n### Response:\n'
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

with torch.no_grad():
    output_ids = trainer.model.generate(**inputs, max_new_tokens=64, temperature=0.7)

print(tokenizer.decode(output_ids[0], skip_special_tokens=True))